# Document Compare

In [1]:
import os

In [2]:
from  langchain_groq import ChatGroq 

In [3]:
llm_model = ChatGroq(model="qwen/qwen3-32b")

In [4]:
llm_model.invoke("weather in coimbatore, india tody ?")

AIMessage(content='<think>\nOkay, the user is asking for the weather in Coimbatore, India today. Let me check how to get that information. First, I should mention that my knowledge is up to December 2023, so I might not have the exact current data. I need to apologize for that. Then, I should suggest reliable sources like weather apps or websites such as Weather.com, AccuWeather, or the India Meteorological Department. Maybe also mention checking their website or a trusted news source. Let me make sure the advice is clear and helpful. Also, maybe add a tip about searching for Coimbatore\'s current weather on Google. Keep it friendly and concise.\n</think>\n\nI don\'t have access to real-time weather data, so I can\'t provide the current weather for Coimbatore, India. My knowledge is up to December 2023, and weather conditions can vary quickly. For the most accurate and up-to-date information, I recommend checking:\n\n1. **Weather apps**: Use services like **Weather.com**, **AccuWeather

In [5]:
document_compare_prompt_template="""
You will be provided 2 versions of same PDF.  You tasks are as follows:

1. Identify the difference between versions
2. Make a note of pdf page and line number
3. Print the difference along with in which page and line number you find the difference

Input  document :

{combined_documents}

Your response should follow this format

{format_instruction}

"""

In [8]:
from langchain_core.prompts import PromptTemplate

In [7]:
compare_prompt = PromptTemplate(
    template=document_compare_prompt_template,
    input_variables=["combined_documents","format_instruction"]
)

In [3]:
from pydantic import BaseModel, RootModel

In [4]:
class ChangeFormat(BaseModel):
    pageno: int
    lineno: int
    changes:str

In [5]:
class SummaryResponse(RootModel[list[ChangeFormat]]):
    pass

In [1]:
from langchain_core.output_parsers import JsonOutputParser


In [6]:
parser = JsonOutputParser(pydantic_object=SummaryResponse)


In [82]:
from langchain.output_parsers import OutputFixingParser


In [83]:
fixing_parser = OutputFixingParser.from_llm(parser=parser, llm=llm_model)


In [84]:
chain = compare_prompt | llm_model | fixing_parser 

In [39]:
from  pathlib import Path

In [43]:
doc_path = Path(os.getcwd()) / "data/compare"

In [52]:
import  fitz


In [68]:
def readpdf(filename):
    with fitz.open(filename) as doc:
        if doc.is_encrypted:
            raise(f"file {filename} is encrypted")
        all_text=[]
        for pagenum in range(doc.page_count):
            page = doc.load_page(pagenum)
            text = page.get_text()
            if text.strip():
                all_text.append(f"\n -- page {pagenum} -- \n {text}")
        return "\n".join(all_text)

In [75]:
content_dict ={}
for filename in sorted(doc_path.iterdir()):
    if filename.is_file() and filename.suffix==".pdf":
        content_dict[filename.name] = readpdf(filename)

combined_documents=[]
for filename,content in content_dict.items():
    combined_documents.append(f"Documents: {filename} \n {content}")
    

        


In [77]:
inputs = {
    "combined_documents" : combined_documents,
    "format_instruction" : parser.get_format_instructions()
}

In [85]:
response =chain.invoke(inputs)

In [86]:
print(response)

[{'pageno': 0, 'lineno': 4, 'changes': "STATUS changed from 'NORMAL' to 'OFF SPEC'"}, {'pageno': 0, 'lineno': 49, 'changes': "Test Result changed from '0.1' to '0.7'"}]


In [87]:
import pandas as pd


In [88]:
df=pd.DataFrame(response)

In [89]:
df

,pageno,lineno,changes
0,0,4,STATUS changed from 'NORMAL' to 'OFF SPEC'
1,0,49,Test Result changed from '0.1' to '0.7'
